# Scattering from a potential

In [ ]:
pip install numpy scipy ipywidgets matplotlib

In [ ]:
import numpy as np
from scipy.integrate import solve_bvp, solve_ivp
import scipy as sp
from scipy.signal import lombscargle
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
xmin = -50
xmax = 70
xs = np.linspace(xmin, xmax, 2500)

def V(x):
#     return 0.01*(x**2 * (np.exp(0.03/(0.01+((x/36)**6))) - 1))
    return -4*np.exp(-(x/4)**2)

def scatter(k):
    x_guess = np.linspace(xmin, xmax)
    y_guess = np.ones([2, len(x_guess)], dtype=np.complex)
           
    def fun(x, y):
        return [y[1], 2*V(x)*y[0] - 2j*k*y[1]]

    def bcs(ya, yb):
        return [yb[0]-1, yb[1]]

    sol = solve_bvp(fun, bcs, x_guess, y_guess)
    fs = sol.sol(xs)[0]
    # transmission coefficient
    T = np.abs(np.mean((fs[xs < -30])))**-1 
    
    # normalise
    fs = fs*T
    psis = fs * np.exp(1j * k * xs)
    return fs, psis, T
    
    
@interact(k=widgets.FloatSlider(min=0, max=24, value=3.5, continuous_update=False))
def scattering(k):
    fs, psis, T = scatter(k)
    
    fig = plt.figure(figsize=(14, 8))
    ax = fig.add_subplot(211)
    E = k**2 / 2
    scale = 0.1 * max(abs(V(xs))) / max(abs(psis))
#     scale = 1
    ax.plot(xs, E + scale*np.real(psis), 'm-',
         xs, V(xs), 'k-',
         xs, E + xs*0, 'k--')

    ax = fig.add_subplot(212)
    ax.plot(xs, np.abs(psis)**2, 'm-')
    ax.set_ylim([0, None])

    ax.set_title('|T| = %.3f' % T)    
#     ax = fig.add_subplot(313)
#     ax.plot(xs, np.abs(fs), 'm-')

    plt.show()
    
#     plt.plot(xs, psis / np.exp(1j*k*sol.x))
#     plt.show()

In [ ]:
ks = np.linspace(0, 24)
Ts = [scatter(k)[2] for k in ks]

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()
ax.plot(ks, Ts)
ax.grid()
ax.set_xlabel('k')
ax.set_ylabel('T')
ax.set_ylim([-.1, 1.3])